<br><h1>Fake News Detection Part 3</h1><br>

In [1]:
import polars as pl
import numpy as np

# Read in data
df = pl.read_csv("../news/data/995,000_rows_preprocessed.csv", columns=["type", "content"])
df.head()

type,content
str,str
"""political""","""plus one articl googl plus tha…"
"""fake""","""cost best senat bank committe …"
"""satire""","""man awoken <NUM> -year coma co…"
"""reliable""","""julia geist ask draw pictur co…"
"""conspiracy""","""<NUM> compil studi vaccin dang…"


In [2]:
# All types
with pl.Config(tbl_rows=-1):
    print(df["type"].value_counts().sort("count", descending=True))

shape: (14, 2)
┌────────────────────────────┬────────┐
│ type                       ┆ count  │
│ ---                        ┆ ---    │
│ str                        ┆ u32    │
╞════════════════════════════╪════════╡
│ reliable                   ┆ 218564 │
│ political                  ┆ 194518 │
│ bias                       ┆ 133232 │
│ fake                       ┆ 104883 │
│ conspiracy                 ┆ 97314  │
│ rumor                      ┆ 56445  │
│ nan                        ┆ 47786  │
│ unknown                    ┆ 43534  │
│ unreliable                 ┆ 35332  │
│ clickbait                  ┆ 27412  │
│ junksci                    ┆ 14040  │
│ satire                     ┆ 13160  │
│ hate                       ┆ 8779   │
│ 2018-02-10 13:43:39.521661 ┆ 1      │
└────────────────────────────┴────────┘


In [3]:
real_labels = ['reliable']
# fake_labels = fake, conspiracy, rumor, unreliable, juncsci, satire, hate
drop_labels = ['unknown', 'nan']
# This could effect real world use on data containing biased, satire etc. articles
# Also makes the training set smaller

df = df.filter(
    ~pl.col("type").is_in(drop_labels)).with_columns(
    pl.when(pl.col("type").is_in(real_labels))
    .then(pl.lit("real"))
    .otherwise(pl.lit("fake"))
    .alias("label")
)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer as _TV

n = df.shape[0]
indices = np.arange(n)
labels = df["label"].to_numpy()

# First split: 80 % train, 20 % temp (will become val + test)
train_idx, temp_idx = train_test_split(
    indices, test_size=0.2, random_state=1, stratify=labels
)
# Second split: halve the temp set into 10 % val + 10 % test
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=1, stratify=labels[temp_idx]
)

df_train = df[train_idx.tolist()]
df_val = df[val_idx.tolist()]
df_test = df[test_idx.tolist()]

print(f"Train: {df_train.shape[0]}  ({df_train.shape[0] / n:.0%})")
print(f"Val:   {df_val.shape[0]}  ({df_val.shape[0] / n:.0%})")
print(f"Test:  {df_test.shape[0]}  ({df_test.shape[0] / n:.0%})")


Train: 722944  (80%)
Val:   90368  (10%)
Test:  90368  (10%)


In [5]:
# Verify that stratified splitting preserved the label ratio
for name, split in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    dist = (
        split.group_by("label")
        .agg(pl.len().alias("count"))
        .sort("label")
        .with_columns(
            (pl.col("count").cast(pl.Float64) / pl.col("count").sum() * 100)
            .round(1)
            .alias("pct")
        )
    )
    print(f"\n{name} split:")
    print(dist)


Train split:
shape: (2, 3)
┌───────┬────────┬──────┐
│ label ┆ count  ┆ pct  │
│ ---   ┆ ---    ┆ ---  │
│ str   ┆ u32    ┆ f64  │
╞═══════╪════════╪══════╡
│ fake  ┆ 548093 ┆ 75.8 │
│ real  ┆ 174851 ┆ 24.2 │
└───────┴────────┴──────┘

Val split:
shape: (2, 3)
┌───────┬───────┬──────┐
│ label ┆ count ┆ pct  │
│ ---   ┆ ---   ┆ ---  │
│ str   ┆ u32   ┆ f64  │
╞═══════╪═══════╪══════╡
│ fake  ┆ 68512 ┆ 75.8 │
│ real  ┆ 21856 ┆ 24.2 │
└───────┴───────┴──────┘

Test split:
shape: (2, 3)
┌───────┬───────┬──────┐
│ label ┆ count ┆ pct  │
│ ---   ┆ ---   ┆ ---  │
│ str   ┆ u32   ┆ f64  │
╞═══════╪═══════╪══════╡
│ fake  ┆ 68511 ┆ 75.8 │
│ real  ┆ 21857 ┆ 24.2 │
└───────┴───────┴──────┘


In [6]:
import gc
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.pipeline import make_pipeline

# HashingVectorizer (much faster and less memory, but we can't look at the words after)
hasher = HashingVectorizer(
    n_features=100_000,            # max_features
    ngram_range=(1, 2),            # unigrams and bigrams
    strip_accents="unicode",       # normalise accented characters
    token_pattern=r"[a-zA-Z]{2,}", # only alphabetic tokens ≥ 2 chars
    alternate_sign=False           # keep non-negative (like standard TF-IDF)
)

# Configure the TF-IDF transformer
tfidf = TfidfTransformer(sublinear_tf=True) # apply 1 + log(tf)

# Combine into a pipeline that acts exactly like TfidfVectorizer
vectorizer = make_pipeline(hasher, tfidf)

# Extract labels before we start deleting things
y_train = (df_train["label"] == "fake").cast(pl.Int8).to_numpy()
y_val   = (df_val["label"]   == "fake").cast(pl.Int8).to_numpy()
y_test  = (df_test["label"]  == "fake").cast(pl.Int8).to_numpy()

# Vectorize train, then drop the df
tfidf_train = vectorizer.fit_transform(df_train["content"])
del df_train; gc.collect()

tfidf_val = vectorizer.transform(df_val["content"])
del df_val; gc.collect()

tfidf_test = vectorizer.transform(df_test["content"])
del df_test; gc.collect()

# Also drop the full df if still around (Let the memory be freed!!)
try:
    del df; gc.collect()
except NameError:
    pass

print(f"TF-IDF train: {tfidf_train.shape}")
print(f"TF-IDF val:   {tfidf_val.shape}")
print(f"TF-IDF test:  {tfidf_test.shape}")

TF-IDF train: (722944, 100000)
TF-IDF val:   (90368, 100000)
TF-IDF test:  (90368, 100000)


In [7]:
# from sklearn.decomposition import PCA
# pca = PCA(n_components=1024)
# X_train = pca.fit_transform(tfidf_train)
# X_val = pca.transform(tfidf_val)
# X_test = pca.transform(tfidf_test)

In [8]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Try a logarithmic sweep of C values and pick the best one on the validation set
c_values = np.logspace(-4, 1, 8)
tuning_rows = []
best_c = None
best_f1 = -1.0

for c_value in c_values:
    candidate = LinearSVC(C=float(c_value), max_iter=10_000, random_state=1, dual="auto")
    candidate.fit(tfidf_train, y_train)

    y_val_candidate = candidate.predict(tfidf_val)
    val_accuracy = accuracy_score(y_val, y_val_candidate)
    val_f1 = f1_score(y_val, y_val_candidate)

    tuning_rows.append({
        "C": float(c_value),
        "val_accuracy": val_accuracy,
        "val_f1": val_f1,
    })
    print(f"{c_value} done")
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_c = float(c_value)

tuning_results = (
    pl.DataFrame(tuning_rows)
    .sort(["val_f1", "val_accuracy"], descending=[True, True])
)
print(tuning_results)


# Refit one final model using the selected C
svm = LinearSVC(C=best_c, max_iter=10_000, random_state=1, dual="auto")
svm.fit(tfidf_train, y_train)

y_val_pred = svm.predict(tfidf_val)
print(f"\nBest C: {best_c}")
print("Validation:")
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"F1: {f1_score(y_val, y_val_pred):.4f}")
print(classification_report(y_val, y_val_pred, target_names=["real", "fake"]))

shape: (8, 3)
┌──────────┬──────────────┬──────────┐
│ C        ┆ val_accuracy ┆ val_f1   │
│ ---      ┆ ---          ┆ ---      │
│ f64      ┆ f64          ┆ f64      │
╞══════════╪══════════════╪══════════╡
│ 0.372759 ┆ 0.961458     ┆ 0.974876 │
│ 0.071969 ┆ 0.958746     ┆ 0.973302 │
│ 1.930698 ┆ 0.955814     ┆ 0.971037 │
│ 10.0     ┆ 0.947647     ┆ 0.96553  │
│ 0.013895 ┆ 0.943243     ┆ 0.963771 │
│ 0.002683 ┆ 0.905398     ┆ 0.941232 │
│ 0.000518 ┆ 0.842776     ┆ 0.906052 │
│ 0.0001   ┆ 0.770217     ┆ 0.8684   │
└──────────┴──────────────┴──────────┘

Best C: 0.3727593720314942
Validation:
Accuracy: 0.9615
F1: 0.9749
              precision    recall  f1-score   support

        real       0.95      0.88      0.92     21856
        fake       0.96      0.99      0.97     68512

    accuracy                           0.96     90368
   macro avg       0.96      0.93      0.95     90368
weighted avg       0.96      0.96      0.96     90368



In [9]:
import joblib

# Save model and vectorizer
joblib.dump(svm, 'svm_tfidf_ngram12_100k.joblib')

joblib.dump(vectorizer, 'vectorizer_tfidf_ngram12_100k.joblib')

['vectorizer_tfidf_ngram12_100k.joblib']